In [ ]:
# --- Load Required Libraries ---
# Ensure necessary packages are installed
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
if (!requireNamespace("remotes", quietly = TRUE))
    install.packages("remotes")
if (!requireNamespace("readr", quietly = TRUE))
    install.packages("readr")
if (!requireNamespace("dplyr", quietly = TRUE))
    install.packages("dplyr")
if (!requireNamespace("tidyverse", quietly = TRUE))
    install.packages("tidyverse")
if (!requireNamespace("MungeSumstats", quietly = TRUE))
    BiocManager::install('MungeSumstats')
if (!requireNamespace("GWASBrewer", quietly = TRUE)) {
    # Need remotes/devtools for GitHub installs
    if (!requireNamespace("remotes", quietly = TRUE)) install.packages("remotes")
    devtools::install_github("jean997/GWASBrewer", 
                         ref = "fdea10e7c4d86585a42c918a2ab828103e07b5ae",
                         build_vignettes = TRUE)
}
if (!requireNamespace("janitor", quietly = TRUE))
    install.packages("janitor")
if (!requireNamespace("SNPlocs.Hsapiens.dbSNP155.GRCh38", quietly = TRUE))
    BiocManager::install("SNPlocs.Hsapiens.dbSNP155.GRCh38")
if (!requireNamespace("BSgenome.Hsapiens.NCBI.GRCh38", quietly = TRUE))
    BiocManager::install("BSgenome.Hsapiens.NCBI.GRCh38")
if (!requireNamespace("arrow", quietly = TRUE))
    install.packages("arrow") # For reading Parquet files

library(readr)
library(dplyr)
library(tidyverse)
library(MungeSumstats)
library(GWASBrewer)
library(janitor)
library(nanoparquet) # Load arrow package

In [ ]:

# --- Configuration ---
# Define the base path for all generated files and directories
base_path <- "PGRS_Average" # You can change "PGRS_1" to any desired base directory name

# Define the location of the input Parquet file (adjust path if necessary)
parquet_file <- "person_ids.parquet" #"final_data_v8_4_22_no_daily.parquet" 
# Ensure this file exists in your working directory or provide the full path.

# Define PGS Score File URL and standardized names
pgs_url <- "https://ftp.ebi.ac.uk/pub/databases/spot/pgs/scores/PGS002196/ScoringFiles/PGS002196.txt.gz" # Using PGS002196 as per original download link
pgs_gz_file <- file.path(base_path, "resources", "pgs_score_raw.txt.gz")
pgs_unzipped_file <- file.path(base_path, "resources", "pgs_score_unzipped.txt") # Intermediate, can be removed later if needed
pgs_no_header_file <- file.path(base_path, "data", "pgs_score_no_header.txt") # Used as input for PLINK score

# --- Create Necessary Directories ---
cat("Creating required directories under:", base_path, "\n")
dir.create(base_path, showWarnings = FALSE)
dir.create(file.path(base_path, "resources"), showWarnings = FALSE)
dir.create(file.path(base_path, "data"), showWarnings = FALSE)
dir.create(file.path(base_path, "plinkfiles"), showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(base_path, "plinkfiles", "filter"), showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(base_path, "plinkfiles", "filtered"), showWarnings = FALSE, recursive = TRUE) # For intermediate PLINK outputs
dir.create(file.path(base_path, "plinkfiles", "habshd"), showWarnings = FALSE, recursive = TRUE)   # For intermediate PLINK outputs
dir.create(file.path(base_path, "plinkfiles", "all"), showWarnings = FALSE, recursive = TRUE)      # For merged PLINK files
dir.create(file.path(base_path, "plinkfiles", "PGRS"), showWarnings = FALSE, recursive = TRUE)    # For final PGRS score output


In [ ]:
# --- Download and Prepare PGS Score File ---
cat("Downloading PGS score file...\n")
# Check if file exists before downloading
if (!file.exists(pgs_gz_file)) {
  download.file(pgs_url, destfile = pgs_gz_file)
  cat("Download complete.\n")
} else {
  cat("PGS score file already exists:", pgs_gz_file, "\n")
}

cat("Unzipping and preparing PGS score file...\n")
# Unzip the file using R's built-in gzfile function
gzfile_con <- gzfile(pgs_gz_file, "r")
unzipped_lines <- readLines(gzfile_con)
close(gzfile_con)

# Write unzipped content to an intermediate file (optional step, could process directly)
# writeLines(unzipped_lines, pgs_unzipped_file)

# Remove header lines (starting with '#') and save to the final input file
lines_no_header <- unzipped_lines[!grepl("^#", unzipped_lines)]
writeLines(lines_no_header, pgs_no_header_file)
cat("PGS score file prepared (header removed):", pgs_no_header_file, "\n")

# Clean up intermediate unzipped lines object
rm(unzipped_lines, lines_no_header)
gc()

# --- Obtain Ancestry Data ---
ancestry_dest_path <- file.path(base_path, "plinkfiles", "filter", "ancestry_preds.tsv")
cat("Checking/Downloading Ancestry data...\n")
if (!file.exists(ancestry_dest_path)) {
  # Now, execute the gsutil cp command
  gsutil_ancestry_args <- c("-u", Sys.getenv("GOOGLE_PROJECT"),
                            "cp",
                            "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv",
                            ancestry_dest_path)
  cat("Running gsutil command:", paste("gsutil", paste(gsutil_ancestry_args, collapse=" ")), "\n")                      
  result_ancestry <- system2("gsutil", 
                             args = gsutil_ancestry_args,
                             stdout = TRUE,
                             stderr = TRUE)
  # Check result (basic check: does the file exist now?)
  if (!file.exists(ancestry_dest_path)) {
      stop("Failed to download ancestry data. gsutil output:\n", paste(result_ancestry, collapse="\n"))
  } else {
      cat("Ancestry data downloaded successfully.\n")
  }
} else {
  cat("Ancestry data already exists:", ancestry_dest_path, "\n")
}



In [ ]:

# --- Prepare Patient List ---
cat("Preparing patient list...\n")

# Step 1: Load unique person_ids from the Parquet file
cat("Loading unique person IDs from:", parquet_file, "\n")
if (!file.exists(parquet_file)) {
    stop("Parquet file not found: ", parquet_file)
}
all_data <- read_parquet(parquet_file, col_select = "person_id")
person_ids_df <- data.frame(person_id = unique(all_data$person_id)) # Ensure it's a dataframe
cat("Loaded", nrow(person_ids_df), "unique person IDs.\n")
# Remove large object immediately and call garbage collector
rm(all_data)
gc() 
cat("Removed large Parquet object from memory.\n")


In [ ]:
# Step 2: Read the chr21.fam file (Assuming this is a template or one example)
# Note: This uses chr21.fam for structure, but filters based on all unique person IDs and EUR ancestry.
# Ensure chr21.fam is available or adjust logic if needed.
# Let's assume it needs downloading like other chromosome files.
chr21_fam_local_path <- file.path(base_path, "plinkfiles", "chr21.fam") # Store it in base plinkfiles dir
chr21_fam_gcs_path <- "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/clinvar/plink_bed/chr21.fam" # Example GCS path

cat("Checking/Downloading chr21.fam for patient list filtering...\n")
if (!file.exists(chr21_fam_local_path)) {
  gsutil_fam_args <- c("-u", Sys.getenv("GOOGLE_PROJECT"),
                       "cp",
                       chr21_fam_gcs_path,
                       chr21_fam_local_path)
  cat("Running gsutil command:", paste("gsutil", paste(gsutil_fam_args, collapse=" ")), "\n")                       
  result_fam <- system2("gsutil", 
                        args = gsutil_fam_args,
                        stdout = TRUE,
                        stderr = TRUE)
  if (!file.exists(chr21_fam_local_path)) {
      stop("Failed to download chr21.fam. gsutil output:\n", paste(result_fam, collapse="\n"))
  } else {
      cat("chr21.fam downloaded successfully.\n")
  }
} else {
  cat("chr21.fam already exists:", chr21_fam_local_path, "\n")
}

fam_data <- read_table(chr21_fam_local_path, col_names = FALSE)
colnames(fam_data) <- c("FID", "IID", "PID", "MID", "SEX", "PHENOTYPE")

# Load ancestry data
ancestry <- read_tsv(ancestry_dest_path)

# Filter only ancestry = eur 
eur_ids <- ancestry %>%
  filter(ancestry_pred == "eur") %>%
  pull(research_id)

# Step 3: Filter fam_data: Keep IIDs present in both EUR ancestry list AND the unique person_id list from Parquet
filtered_fam <- fam_data[fam_data$IID %in% eur_ids & fam_data$IID %in% person_ids_df$person_id, ]

# Step 4: Create the patient_list dataframe with FID and IID
patient_list <- filtered_fam[, c("FID", "IID")]

# Step 5: Write the patient_list to a file
patient_list_file <- file.path(base_path, "plinkfiles", "filter", "patient_list.txt")
write_tsv(patient_list, patient_list_file, col_names = FALSE)

cat("Patient list created with", nrow(patient_list), "individuals (EUR ancestry and in Parquet data).\n")
cat("File saved as:", patient_list_file, "\n")

# Clean up intermediate objects
rm(fam_data, ancestry, eur_ids, filtered_fam, person_ids_df)
gc()

In [ ]:
# Define the list of chromosomes you want to process
chromosomes <- c("chr1", "chr2", "chr3", "chr4", "chr5", "chr6", "chr7", "chr8", "chr9", "chr10",
                 "chr11", "chr12", "chr13", "chr14", "chr15", "chr16", "chr17", "chr18", "chr19", "chr20",
                 "chr21", "chr22", "chrX", "chrY")


#chromosomes <- c( "chr22")
# Function to process a single chromosome
process_chromosome <- function(chr, base_path, pgs_input_file, patient_list_path) {
  cat("Starting processing for", chr, "\n")

  # Define subdirectories
  raw_dir <- file.path(base_path)
  filter_dir <- file.path(base_path, "filter")
  filtered_dir <- file.path(base_path, "filtered")
  habshd_dir <- file.path(base_path, "habshd")

  # Create directories
  dir.create(filter_dir, showWarnings = FALSE, recursive = TRUE)
  dir.create(filtered_dir, showWarnings = FALSE, recursive = TRUE)
  dir.create(habshd_dir, showWarnings = FALSE, recursive = TRUE)

  # Download files
  cat("Downloading files for", chr, "\n")
  file_types <- c("bim", "bed", "fam")
  for (file_type in file_types) {
    cat("  Downloading", file_type, "file\n")
    dest_file <- file.path(raw_dir, paste0(chr, ".", file_type))
    source_file <- paste0("gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/clinvar/plink_bed/", chr, ".", file_type)
    
    result <- system2("gsutil",
                      args = c("-m","-u", Sys.getenv("GOOGLE_PROJECT"),
                               "cp",
                               source_file,
                               dest_file),
                      stdout = TRUE,
                      stderr = TRUE)
    # Add basic error check for download
    if (!file.exists(dest_file)) {
       warning("Failed to download ", dest_file, " from ", source_file)
       # Consider stopping execution if download fails: stop("Download failed for ", chr)
    }
    cat("  Download complete\n")
  }

  # Process .bim file
  cat("Processing .bim file for", chr, "\n")
  bim_file_path <- file.path(raw_dir, paste0(chr, ".bim"))
  bim.raw <- readr::read_tsv(bim_file_path, col_names = FALSE, show_col_types = FALSE) %>%
    janitor::clean_names() %>%
    dplyr::rename(chrom = x1, id = x2, cm = x3, pos = x4, ref = x6, alt = x5)
  cat("  .bim file processed\n")

  # Simulate GWAS data
  cat("Simulating GWAS data for", chr, "\n")
  # Ensure GWASBrewer is available and loaded
  if (!requireNamespace("GWASBrewer", quietly = TRUE)) {
      stop("Package 'GWASBrewer' is required but not installed.")
  }
  dat_simple <- GWASBrewer::sim_mv(G = 1, N = 2578, J = nrow(bim.raw), h2 = 0.5, pi = 0.01, est_s = TRUE, af = function(n){rbeta(n, 1, 5)})
  cat("  GWAS data simulation complete\n")

  bim <- dplyr::bind_cols(
    bim.raw, dat_simple$beta_hat, dat_simple$se_beta_hat, dat_simple$snp_info
  ) %>%
    tibble::as_tibble() %>%
    dplyr::rename(beta = ...7, se = ...8) %>% # Adjusted renaming based on typical bind_cols behavior
    dplyr::select(-SNP) %>%
    dplyr::mutate(
      Z = beta / se,
      P = 2 * (1 - stats::pt(abs(Z), Inf)) # Use stats::pt for clarity
    )

  # MungeSumstats
  cat("Running MungeSumstats for", chr, "\n")
  # Ensure MungeSumstats is available and loaded
  if (!requireNamespace("MungeSumstats", quietly = TRUE)) {
      stop("Package 'MungeSumstats' is required but not installed.")
  }
  # Check if input 'bim' has required columns for MungeSumstats
  required_cols <- c("id", "chrom", "pos", "alt", "ref", "beta", "se", "P")
  if (!all(required_cols %in% names(bim))) {
      stop("Input data frame 'bim' is missing required columns for MungeSumstats. Found: ", paste(names(bim), collapse=", "))
  }
  
  # Create a temporary file for MungeSumstats if needed, or pass data directly
  # MungeSumstats::format_sumstats can accept a data.frame directly
  reformatted <- MungeSumstats::format_sumstats(path = bim, # Pass the data frame directly
                                                ref_genome = "GRCh38",
                                                return_data = TRUE,
                                                bi_allelic_filter = FALSE, # Explicitly set to FALSE
                                                N_dropNA = FALSE, # Explicitly set to FALSE
                                                rmv_chr = NULL,   # Explicitly set to NULL
                                                allele_flip_check = FALSE,
                                                allele_flip_drop = FALSE,
                                                allele_flip_frq = FALSE,
                                                dbSNP = 155) %>%
    tibble::as_tibble()
  cat("  MungeSumstats complete\n")

  rsid_test_file <- file.path(filter_dir, paste0(chr, "_rsid_test.txt"))
  reformatted %>%
    dplyr::select(ID, SNP) %>% # Assuming 'ID' is the original ID and 'SNP' is the updated rsID from MungeSumstats
    readr::write_tsv(rsid_test_file, col_names = FALSE)

  # PLINK2 command
  cat("Running PLINK2 command for", chr, "\n")
  plink2_args <- c(
    "--bfile", file.path(raw_dir, chr),
    "--keep", patient_list_path, # Use parameterized patient list path
    "--geno", "0.05",
    "--snps-only", "just-acgt", # Often recommended with --snps-only
    "--rm-dup", "exclude-all",
    "--update-name", rsid_test_file, # Path to the rsID update file
    "--keep-allele-order",
    "--make-bed",
    "--out", file.path(filtered_dir, chr) # Output to filtered directory
  )
  result_plink2 <- system2("plink2", args = plink2_args, stdout = TRUE, stderr = TRUE)
  # Add basic check for plink2 output files
  if (!file.exists(paste0(file.path(filtered_dir, chr), ".bed"))) {
      warning("PLINK2 output file not found for ", chr, ". Check PLINK2 log.")
      print(result_plink2) # Print output/error from system2 call
  }
  cat("  PLINK2 command complete\n")

  # Filter by required variants
  cat("Filtering by required variants for", chr, "\n")
  variant_list <- readr::read_tsv(pgs_input_file, show_col_types = FALSE) # Use parameterized PGS input file path
  rsids_to_keep_file <- file.path(filter_dir, "rsids_to_keep.txt") # Place temp file in filter dir
  # Check if rsID column exists before writing
  if (!"rsID" %in% names(variant_list)) {
      stop("Column 'rsID' not found in the file: ", pgs_input_file)
  }
  write.table(variant_list$rsID, file = rsids_to_keep_file, row.names = FALSE, col.names = FALSE, quote = FALSE)

  # PLINK command (using plink 1.9 often for basic filtering)
  cat("Running final PLINK command for", chr, "\n")
  plink_args <- c(
    "--bfile", file.path(filtered_dir, chr), # Input from filtered directory
    "--extract", rsids_to_keep_file,         # Use the rsID list file
    "--make-bed",
    "--out", file.path(habshd_dir, chr)       # Final output to habshd directory
  )
  result_plink1 <- system2("plink", args = plink_args, stdout = TRUE, stderr = TRUE)
   # Add basic check for plink output files
  if (!file.exists(paste0(file.path(habshd_dir, chr), ".bed"))) {
      warning("Final PLINK output file not found for ", chr, ". Check PLINK log.")
      print(result_plink1) # Print output/error from system2 call
  }
  cat("  Final PLINK command complete\n")

  cat("Processing complete for", chr, "\n")
  # Return the path prefix of the final bed file
  return(file.path(habshd_dir, chr))
}


# Function to merge chromosome file with existing 'all' file
merge_chromosome_file <- function(chr_file_prefix, base_path) {
  #all_dir <- file.path(base_path, "all")
  all_dir <- file.path(base_path, "plinkfiles", "all")
  dir.create(all_dir, showWarnings = FALSE, recursive = TRUE)
  all_file_prefix <- file.path(all_dir, "all_chromosomes")

  # Check if the input chromosome file exists before proceeding
  if (!file.exists(paste0(chr_file_prefix, ".bed"))) {
    warning("Input file for merging not found: ", paste0(chr_file_prefix, ".bed"))
    return() # Stop merging if input is missing
  }

  if (!file.exists(paste0(all_file_prefix, ".bed"))) {
    cat("Creating initial 'all' file from", basename(chr_file_prefix), "\n")
    # If 'all' file doesn't exist, just copy the chromosome file components
    file.copy(paste0(chr_file_prefix, ".bed"), paste0(all_file_prefix, ".bed"), overwrite = TRUE)
    file.copy(paste0(chr_file_prefix, ".bim"), paste0(all_file_prefix, ".bim"), overwrite = TRUE)
    file.copy(paste0(chr_file_prefix, ".fam"), paste0(all_file_prefix, ".fam"), overwrite = TRUE)
    cat("Initial 'all' file created\n")
  } else {
    cat("Merging", basename(chr_file_prefix), "with existing 'all' file\n")
    # Define temporary output prefix for merging
    temp_merge_prefix <- paste0(all_file_prefix, "_temp_merge")

    # Construct plink merge command arguments
    merge_args <- c(
      "--bfile", all_file_prefix,     # Existing merged file
      "--bmerge", chr_file_prefix,    # New chromosome file to merge
      "--make-bed",
      "--out", temp_merge_prefix      # Temporary output prefix
    )

    merge_command <- "plink" # Assuming plink 1.9 for merging
    result <- system2(merge_command, args = merge_args, stdout = TRUE, stderr = TRUE)
    cat("Merge command intermediate step complete\n")

    # Check if merge was successful (plink creates .bed, .bim, .fam)
    if (file.exists(paste0(temp_merge_prefix, ".bed"))) {
        # Replace old 'all' files with new merged files
        file.rename(paste0(temp_merge_prefix, ".bed"), paste0(all_file_prefix, ".bed"))
        file.rename(paste0(temp_merge_prefix, ".bim"), paste0(all_file_prefix, ".bim"))
        file.rename(paste0(temp_merge_prefix, ".fam"), paste0(all_file_prefix, ".fam"))
        # Clean up temporary log/nosex files if they exist
        if (file.exists(paste0(temp_merge_prefix, ".log"))) file.remove(paste0(temp_merge_prefix, ".log"))
        if (file.exists(paste0(temp_merge_prefix, ".nosex"))) file.remove(paste0(temp_merge_prefix, ".nosex"))
         # Check for plink's automatic handling of mismatched SNPs (-merge-list)
        if (file.exists(paste0(temp_merge_prefix, "-merge.missnp"))) file.remove(paste0(temp_merge_prefix, "-merge.missnp"))
        cat("'all' file updated successfully\n")
    } else {
        warning("PLINK merge failed. Check output/log:\n", paste(result, collapse="\n"))
        # Clean up potential merge failure artifacts like -merge.missnp
        if (file.exists(paste0(all_file_prefix, "-merge.missnp"))) {
             cat("Attempting to handle merge failure (e.g. mismatched SNPs). Check ", paste0(all_file_prefix, "-merge.missnp"), "\n")
             # NOTE: Automatic handling of merge failures is complex.
             # User might need to manually intervene based on plink output/log.
             # Basic cleanup: remove the missnp file if it exists from a failed direct merge attempt
             file.remove(paste0(all_file_prefix, "-merge.missnp"))
        }
         # Clean up potential temp files even on failure
        if (file.exists(paste0(temp_merge_prefix, ".log"))) file.remove(paste0(temp_merge_prefix, ".log"))
        if (file.exists(paste0(temp_merge_prefix, ".nosex"))) file.remove(paste0(temp_merge_prefix, ".nosex"))
    }
  }
}


# Function to clean up intermediate chromosome files
clean_up_files <- function(chr, base_path) {
  cat("Cleaning up intermediate files for", chr, "\n")

  # Define expected file prefixes and directories
  raw_prefix <- file.path(base_path, chr)
  filtered_prefix <- file.path(base_path, "filtered", chr)
  habshd_prefix <- file.path(base_path, "habshd", chr)
  filter_files <- c(
      file.path(base_path, "filter", paste0(chr, "_rsid_test.txt"))
      # Note: rsids_to_keep.txt is common, maybe don't delete it here? Or handle outside loop.
      # file.path(base_path, "filter", "rsids_to_keep.txt")
  )

  # List of file extensions to remove for plink file sets
  extensions <- c(".bed", ".bim", ".fam", ".log", ".nosex") # Include common plink output files

  # Function to safely remove a file if it exists
  safe_remove <- function(file_path) {
    if (file.exists(file_path)) {
      file.remove(file_path)
      # cat("  Removed:", file_path, "\n") # Optional: uncomment for verbose cleanup
    }
  }

  # Remove files for each prefix and extension combination
  for (prefix in c(raw_prefix, filtered_prefix, habshd_prefix)) {
    for (ext in extensions) {
      safe_remove(paste0(prefix, ext))
    }
  }

  # Remove specific filter files
  for (f_file in filter_files) {
      safe_remove(f_file)
  }

  # Optional: Remove plink2 intermediate files if any (.pgen, .pvar, .psam if --make-pgen used)
  # safe_remove(paste0(filtered_prefix,".pgen"))
  # safe_remove(paste0(filtered_prefix,".pvar"))
  # safe_remove(paste0(filtered_prefix,".psam"))

  cat("Cleanup attempt complete for", chr, "\n")
}




In [ ]:
# --- Main Processing Loop ---
cat("\n=== Starting Chromosome Processing ===\n")
total_chromosomes <- length(chromosomes)
all_chromosomes_prefix <- file.path(base_path, "plinkfiles", "all", "all_chromosomes")
final_habshd_files <- list() # To store paths of successfully processed chromosome files


In [ ]:
total_chromosomes

In [ ]:


cat("\n=== Starting Chromosome Processing For Loop===\n")
for (i in seq_along(chromosomes)) {
    cat("\n=== Starting Chromosome Processing inner Loop ===\n")
  chr <- chromosomes[i]
  cat("\n--- Processing chromosome", i, "of", total_chromosomes, ":", chr, "---\n")
  
  # Process the chromosome
  # Pass necessary file paths to the function
  chr_final_prefix <- tryCatch({
      process_chromosome(chr = chr, 
                         base_path = base_path, 
                         pgs_input_file = pgs_no_header_file, 
                         patient_list_path = patient_list_file)
  }, error = function(e) {
      cat("Error processing chromosome", chr, ": ", conditionMessage(e), "\n")
      return(NULL) # Indicate failure
  })

  # If processing was successful, proceed to merge and cleanup
  if (!is.null(chr_final_prefix)) {
      final_habshd_files[[chr]] <- chr_final_prefix # Store prefix if successful
      cat("\nMerging", chr, "into all file\n")
      merge_chromosome_file(chr_final_prefix, base_path)
      
      cat("\nCleaning up intermediate files for", chr, "\n")
      clean_up_files(chr, base_path)
  } else {
      cat("\nSkipping merge and cleanup for failed chromosome:", chr, "\n")
  }

  cat("\n--- Finished processing attempt for", chr, "---\n")
  cat("\nProgress:", i, "out of", total_chromosomes, "chromosomes attempted\n")
}

cat("\n=== Chromosome Processing Loop Finished ===\n")

# Check if the final merged file exists
if (file.exists(paste0(all_chromosomes_prefix, ".bed"))) {
    cat("\nAll successfully processed chromosomes have been merged into:", all_chromosomes_prefix, "\n")

    # --- Calculate Polygenic Risk Score using PLINK ---
    cat("\nCalculating Polygenic Risk Score using the merged file...\n")
    
    # Define PLINK command for scoring
    plink_score_command <- "plink"
    plink_score_out_prefix <- file.path(base_path, "plinkfiles", "PGRS", "temp_PGRS_calc") # Temporary prefix
    final_pgrs_output_file <- file.path(base_path, "plinkfiles", "PGRS", "PGRS.txt") # Final desired name

    # Arguments for --score: 
    # file, rsid col, effect allele col, effect weight col, options (header, sum)
    # Check columns in pgs_no_header_file: Needs rsID, effect_allele, effect_weight
    # Assuming standard PGS Catalog format: rsID=col 2, effect_allele=col 3, effect_weight=col 5 (adjust if different!)
    # Let's re-read the header of the *original* unzipped file to be sure
    gzfile_con_check <- gzfile(pgs_gz_file, "r")
    header_lines <- readLines(gzfile_con_check, n=20) # Read first 20 lines
    close(gzfile_con_check)
    score_header <- header_lines[grepl("^rsID", header_lines) | grepl("^effect_allele", header_lines)] # Find the header line
    print("Detected Score File Header:")
    print(score_header)
    # Based on PGS002196.txt.gz format:
    # #chr_name chr_position rsID effect_allele other_allele effect_weight ...
    # So: rsID=col 3, effect_allele=col 4, effect_weight=col 6
    # HOWEVER, the original script used 1 4 6. Let's stick to that assuming it was correct for the *intended* file (PGS002209?)
    # Let's use 1 4 6 as in the original script, but add a note.
    # **IMPORTANT**: Verify these column numbers (1, 4, 6) are correct for the actual score file used (PGS002196)!
    # Column 1: rsID (Assumed, check file)
    # Column 4: effect_allele (Assumed, check file)
    # Column 6: effect_weight (Assumed, check file)
    
    plink_score_args <- c(
      "--bfile", all_chromosomes_prefix, # Use the final merged file
      "--score", pgs_no_header_file, "1", "4", "6", # Use prepared score file & ASSUMED columns 1,4,6
      "header", # Indicate the score file has a header (though we removed '#' lines, the first data line is header)
      "sum",    # Calculate sum score per individual
      "--out", plink_score_out_prefix # Output prefix for PLINK score results
    )

    cat("Executing: plink", paste(plink_score_args, collapse=" "), "\n")
    result_score <- system2(plink_score_command, args = plink_score_args, stdout = TRUE, stderr = TRUE)

    # Print the output/log from PLINK scoring
    cat("PLINK scoring command output:\n")
    cat(paste(result_score, collapse = "\n"), "\n")

    # Rename the output file (PLINK typically creates .sscore for sum scores)
     original_plink_score_output <- paste0(plink_score_out_prefix, ".profile")
    cat("Checking for PLINK output file:", original_plink_score_output, "\n")
    if (file.exists(original_plink_score_output)) {
      file.rename(original_plink_score_output, final_pgrs_output_file)
      cat("Successfully calculated PGRS. Renamed PLINK output to:", final_pgrs_output_file, "\n")
      # Clean up log file if exists
      log_file <- paste0(plink_score_out_prefix, ".log")
      if(file.exists(log_file)) file.remove(log_file)
      nosex_file <- paste0(plink_score_out_prefix, ".nosex")
       if(file.exists(nosex_file)) file.remove(nosex_file)

    } else {
      cat("Warning: Expected PLINK score output file not found:", original_plink_score_output, "\n")
      cat("Check PLINK output above for errors.\n")
    }

} else {
    cat("\nError: The final merged file", paste0(all_chromosomes_prefix, ".bed"), "was not created. Cannot calculate PGRS.\n")
    cat("Review the logs from the chromosome processing and merging steps.\n")
}


cat("\n--- Script Finished ---\n")


plink_score_command <- "plink"
plink_score_out_prefix <- file.path(base_path, "plinkfiles", "PGRS", "temp_PGRS_calc") # Temporary prefix
    final_pgrs_output_file <- file.path(base_path, "plinkfiles", "PGRS", "PGRS.txt") # Final desired name   
plink_score_args <- c(
      "--bfile", all_chromosomes_prefix, # Use the final merged file
      "--score", pgs_no_header_file, "1", "4", "6", # Use prepared score file & ASSUMED columns 1,4,6
      "header", # Indicate the score file has a header (though we removed '#' lines, the first data line is header)
      "sum",    # Calculate sum score per individual
      "--out", plink_score_out_prefix # Output prefix for PLINK score results
    )

    cat("Executing: plink", paste(plink_score_args, collapse=" "), "\n")
    result_score <- system2(plink_score_command, args = plink_score_args, stdout = TRUE, stderr = TRUE)

In [ ]:
result_score